<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Lab - Building a Supervisor Agent with Agent Bricks

## Overview
In this lab, you will create a **Supervisor Agent** using Agent Bricks and connect it to a pre-built Knowledge Assistant (KA) and Unity Catalog function tools. The Supervisor Agent will route user queries to the KA for document-based questions and to UC function tools for live data operations.

## Learning Objectives
By the end of this lab, you will be able to:
1. Inspect a pre-built Knowledge Assistant and pre-registered UC tools
2. Create a Supervisor Agent via the Agent Bricks UI
3. Attach a Knowledge Assistant as a subagent
4. Attach UC function tools to the Supervisor Agent
5. Test multi-capability routing in AI Playground
6. Understand when to use a Supervisor Agent vs a standalone KA

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: <a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">How to select an environment version</a></li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-8

## B. Inspect Pre-Built Resources

The classroom setup has prepared two types of resources for your Supervisor Agent:

1. **A Knowledge Assistant (KA)**: provisioned by the workspace administrator and indexed on Airbnb listings documentation stored in the shared `dbacademy` catalog. The KA can answer questions about property details, descriptions, and summaries from the listings PDF. In this lab, you have access to this KA but you will not be directly querying it. 

2. **UC Function Tools**: registered in your personal catalog and schema during classroom setup:
   - `avg_neigh_price`: calculates the average listing price for a neighborhood
   - `cnt_by_room_type`: counts listings by room type in a neighborhood
   - `airbnb_posting_info`: fetches detailed information about a specific listing

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">The Knowledge Assistant was created by the workspace administrator during provisioning. You will attach it to your own Supervisor Agent alongside your personal UC function tools. Do not create or modify the KA itself.</p>
</div>
</div>
</div>

### B1. Verify UC Tools via MCP

In [0]:
import nest_asyncio
nest_asyncio.apply()

from databricks.sdk import WorkspaceClient
from databricks_mcp import DatabricksMCPClient

ws = WorkspaceClient()
workspace_host = f"https://{spark.conf.get('spark.databricks.workspaceUrl')}"

mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

tools = mcp_client.list_tools()

print(f"Found {len(tools)} tools in {catalog_name}.{schema_name}:\n")
for tool in tools:
    print(f"  {tool.name}")
    print(f"    {tool.description[:100]}...")
    print()

## C. Create a Supervisor Agent

Now you will create a **Supervisor Agent (SA)** via the Agent Bricks UI. The SA will coordinate between:
- The **Knowledge Assistant** for document-based Q&A (property descriptions, summaries)
- **UC function tools** for live data queries (pricing, room counts, listing details)

### C1. Navigate to Agent Bricks

1. In the Databricks workspace, click **Agents** in the left sidebar
2. Click **Create Agent**
3. Select **Supervisor Agent**

### C2. Configure and Update The Supervisor Agent

1. Attach Tools and sub-agents:
    - Click **Add a Knowledge Assistant** and select the Airbnb Knowledge Assistant. 
        - This initializes your agent
    - Click in the search box that says **Search Databricks** and click **UC functions**.
    - Find the functions `avg_neigh_price`, `cnt_by_room_type`, and `airbnb_posting_info` from the `airbnb_agent` schema in your user catalog. 

2. Add Instructions: Copy and paste the following to add instructions to your SA

<div class="code-block-light" data-language="text">
Respond in a professional tone. Be concise in your answers. 
</div>

3. Add a Description: Copy and paste the following for the description of your agent. 

<div class="code-block-light" data-language="text">
Supervisor agent that routes Airbnb questions to a Knowledge Assistant for document Q&A and UC function tools for live data analysis.
</div>

4. Notice the changes made are saved automatically with a _last saved_ message near the title of your SA.
5. Rename your SA the same as the output from running the next cell. This will help distinguish your agent from other agents in the workspace since it uses your catalog name, which is based your username in the workspace. 

<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

In [0]:
print(f"{catalog_name}-sa")


**Note🚨:** After creating an AI Search Index, wait 10–15 minutes for syncing to complete before testing, even if the endpoint shows the index as attached.

## D. Test the Supervisor Agent

### D1. Test Document Q&A (routes to KA)

Try a question that requires information from the Airbnb listings documents:

<div class="code-block-light" data-language="text">
Tell me about the property descriptions for listings in the Mission district.
</div>
The Supervisor should route this to the Knowledge Assistant, which retrieves information from the indexed PDF.


<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

### D2. Test Live Data Queries (routes to UC tools)

Try a question that requires live data computation:

<div class="code-block-light" data-language="text">
What is the average price for listings in Mission, and how many private rooms are available there?
</div>

The Supervisor should route this to the UC function tools (`avg_neigh_price` and `cnt_by_room_type`).


<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

### D3. Test Combined Routing

Try asking questions in a follow-up way that requires both document knowledge and live data:

<div class="code-block-light" data-language="text">
What can you tell me about listing 958 from the documents.
</div>
<div class="code-block-light" data-language="text">
Also what is the average price in its neighborhood?
</div>

The Supervisor should use both the KA (for document details) and the UC tools (for pricing data).


<style id="code-block-light-css">
.code-block-light {
    position: relative;
    margin: 16px 0;
    background: #f8f9fa;
    border: 1px solid #dee2e6;
    border-radius: 8px;
    padding: 16px 20px;
    padding-right: 80px;
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
    font-size: 0.95em;
    line-height: 1.5;
    color: #333;
    white-space: pre-wrap;
}
</style>

<script>
(function(){
document.querySelectorAll('.code-block-light').forEach(function(block) {
    if (block.getAttribute('data-processed')) return;
    block.setAttribute('data-processed', 'true');
    var code = block.textContent.trim();
    var btn = document.createElement('button');
    btn.textContent = 'Copy';
    btn.style.cssText = 'position:absolute;top:8px;right:8px;padding:4px 12px;font-size:12px;background:#fff;color:#333;border:1px solid #dee2e6;border-radius:4px;cursor:pointer;font-weight:600;';
    btn.onmouseover = function(){ this.style.background='#e9ecef'; };
    btn.onmouseout = function(){ this.style.background='#fff'; };
    btn.onclick = function() {
        var t = document.createElement('textarea');
        t.value = code;
        document.body.appendChild(t);
        t.select();
        document.execCommand('copy');
        document.body.removeChild(t);
        this.textContent = 'Copied!';
        var b = this;
        setTimeout(function(){ b.textContent='Copy'; },2000);
    };
    block.style.position = 'relative';
    block.insertBefore(btn, block.firstChild);
});
})();
</script>

## E. Clean Up

When you are finished testing, delete the **Supervisor Agent** you created:

1. Navigate to **Agents** in the workspace
2. Find your Supervisor Agent
3. Click the three-dot menu (top right)
4. Click **Delete**
5. Confirm deletion

The Knowledge Assistant is a shared for others in the workspace: **do not delete**.

## F. Conclusion

In this lab you created a Supervisor Agent using Agent Bricks that combines a pre-built Knowledge Assistant with UC function tools. You saw how the Supervisor routes queries to the appropriate resource: the KA for document-based questions and UC tools for live data operations.

&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>